In [1]:
import os 
import pandas as pd 
import numpy as np
import tensorflow as tf
import h5py
print(tf.__version__)  # Should show 2.13.0
import keras 
from keras import layers, models, Input
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

2.13.0


In [2]:

df_data = pd.read_csv(filepath_or_buffer='/Users/jawad/Desktop/HarmonAI/team3/server/data/toy_data/cleaned_data.csv')

In [3]:
data_shape = df_data.shape
print(F"rows: {data_shape[0]}")
print(F"columns: {data_shape[1]}")
print(F"rows: {data_shape}")

print(df_data.head(10))

rows: 1417708
columns: 27
rows: (1417708, 27)
   timestamp  chroma_0  chroma_1  chroma_2  chroma_3  chroma_4  chroma_5  \
0   0.000000  0.198482  0.000000  0.000000  0.635556  0.741292  1.004300   
1   0.046440  0.310882  0.000000  0.000000  0.693876  0.628553  1.080040   
2   0.092880  0.404969  0.000000  0.037238  0.682770  0.591140  1.146830   
3   0.139320  0.480218  0.000000  0.005002  0.435639  0.450297  1.211120   
4   0.185760  0.539064  0.146614  0.010891  0.444361  0.196939  1.298150   
5   0.232200  0.165807  0.000000  0.455848  0.965423  0.000000  1.232440   
6   0.278639  0.000000  0.094263  0.576515  1.961290  0.000000  0.766963   
7   0.325079  0.001740  0.021126  0.335464  2.595800  0.000000  0.000000   
8   0.371519  0.000000  0.000000  0.000000  2.822160  0.000000  0.000000   
9   0.417959  0.494383  0.016506  0.102202  2.837510  0.000000  0.000000   

   chroma_6  chroma_7  chroma_8  ...  chroma_16  chroma_17  chroma_18  \
0  0.814440  0.029282  0.141189  ...   1.165

In [4]:
x_data = df_data.drop(columns=['key','quality'])
y_key_data = df_data['key']
y_quality_data = df_data['quality']


#split into train validate and test
x_train, x_test, y_key_train, y_key_test, y_quality_train, y_quality_test = train_test_split(x_data, 
    y_key_data, 
    y_quality_data, 
    test_size=0.2, 
    random_state=42)

print(x_test)
print(x_train)
x_test.to_numpy()
x_train.to_numpy()
y_key_test.to_numpy()
y_key_train.to_numpy()
y_quality_test.to_numpy()
y_quality_train.to_numpy()


#add padding 
padding = 0 


#make it into the correct shape

          timestamp  chroma_0  chroma_1  chroma_2  chroma_3  chroma_4  \
409405   108.065669  0.000000  0.034222  2.364240  0.007879  0.000000   
383496    26.284989  0.006861  0.196629  0.384269  0.000000  0.000000   
541458   185.666757  2.286660  0.126700  0.000000  0.000000  0.000000   
585049   235.682540  0.419340  0.720966  0.514710  0.000000  0.425830   
46719     10.913379  0.000000  0.000000  0.000000  0.000000  0.000000   
...             ...       ...       ...       ...       ...       ...   
1184477  172.431383  0.000000  0.000000  1.517910  0.035666  0.042216   
1039363   70.449342  0.823391  0.843409  0.000000  0.000000  0.000000   
1056114  193.747302  1.972430  0.900847  0.015660  0.839336  0.409984   
61335    232.524626  0.260011  0.807941  1.139190  0.000000  0.000000   
1106265  260.295692  0.000000  0.000000  0.047454  2.295650  0.070573   

         chroma_5  chroma_6  chroma_7  chroma_8  ...  chroma_14  chroma_15  \
409405   0.572059  1.198390  0.691098  0.0000

In [5]:
def paddington(data,padVal, div=100):
    
    reminder = len(data)%div

    if reminder == 0:
        return data

    amount_to_pad = div-reminder

    if data.ndim == 1:
        padded = np.pad(data,(0,amount_to_pad),constant_values=padVal)
    else:
        padded = np.pad(data, ((0, amount_to_pad), (0, 0)), constant_values=padVal) 
    return padded       

In [6]:
x_test_pad = paddington(x_test,padding)
x_train_pad = paddington(x_train,padding)
y_key_test = paddington(y_key_test,padding)
y_key_train = paddington(y_key_train,padding)
y_quality_test = paddington(y_quality_test,padding)
y_quality_train = paddington(y_quality_train,padding)

x_train_batches = x_train_pad.reshape(-1, 100, 25)
x_test_batches  = x_test_pad.reshape(-1, 100, 25)
y_key_train_batches = y_key_train.reshape(-1,100)
y_quality_train_batches = y_quality_train.reshape(-1,100)


In [7]:
inputs = Input(shape=(100,25))
output = layers.Conv1D(filters=24, kernel_size=3,strides=1,padding="same",activation='relu')(inputs)
output = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(output)

keyInput= layers.Dense(32,activation='relu')(output)
key= layers.Dense(14,activation= 'softmax',name='key')(keyInput)

shapeInput = layers.Dense(32,activation='relu')(output)
quality = layers.Dense(4,activation='softmax',name='quality')(shapeInput)

model = models.Model(inputs=inputs, outputs = [key, quality])
model.compile(optimizer='adam',
              loss= 'sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [8]:
model.fit(x_train_batches, {'key': y_key_train_batches, 'quality': y_quality_train_batches}, 
                      batch_size=32, 
                      epochs=5)



Epoch 1/5
355/355 [==============================] - 32s 87ms/step - loss: 2.8188 - key_loss: 1.9942 - quality_loss: 0.8246 - key_accuracy: 0.3664 - quality_accuracy: 0.6735
Epoch 2/5
355/355 [==============================] - 34s 95ms/step - loss: 2.1824 - key_loss: 1.4109 - quality_loss: 0.7715 - key_accuracy: 0.5882 - quality_accuracy: 0.6910
Epoch 3/5
355/355 [==============================] - 34s 97ms/step - loss: 2.1043 - key_loss: 1.3665 - quality_loss: 0.7378 - key_accuracy: 0.5993 - quality_accuracy: 0.7024
Epoch 4/5
355/355 [==============================] - 35s 98ms/step - loss: 2.0584 - key_loss: 1.3421 - quality_loss: 0.7163 - key_accuracy: 0.6057 - quality_accuracy: 0.7137
Epoch 5/5
355/355 [==============================] - 34s 96ms/step - loss: 2.0271 - key_loss: 1.3247 - quality_loss: 0.7024 - key_accuracy: 0.6102 - quality_accuracy: 0.7196


In [9]:
model.save(filepath='/Users/jawad/Desktop/HarmonAI/team3/server/data/output/saved_trained_model.keras')

In [10]:
df_to_pred = pd.read_csv(filepath_or_buffer='/Users/jawad/Desktop/HarmonAI/team3/server/data/toy_data/bothchroma.csv',header=None)

df_to_pred = df_to_pred.drop(columns=[0])
#--------------------------------------------- DEBUG ---------------------------
print(f"to pred shape is {df_to_pred.shape}") # must be (1417800, 25)?
df_to_pred.to_numpy()
#--------------------------------------------- END DEBUG ---------------------------

padded_pred = paddington(df_to_pred,padding)

#--------------------------------------------- DEBUG ---------------------------

print(f"Total elements: {padded_pred.size}") 
print(f"Original shape: {padded_pred.shape}")
#--------------------------------------------- END DEBUG ---------------------------


padded_pred = padded_pred.reshape(-1,100,25)
predictions = model.predict(padded_pred)

key_probs = predictions[0].reshape(-1, predictions[0].shape[-1])
quality_probs = predictions[1].reshape(-1, predictions[1].shape[-1])

keys_final = np.argmax(key_probs,axis=-1)
quality_final = np.argmax(quality_probs,axis= -1)

print(keys_final)

to pred shape is (4442, 25)
Total elements: 112500
Original shape: (4500, 25)
2/2 [==============================] - 0s 16ms/step
[12 12 12 ...  0  0  0]
